# GHSL — building height

Download **GHS-BUILT-H** (average net building height, ANBH) for 2018 at 100 m
over a Lisbon AOI, then render the height raster with matplotlib. GHSL is a
*raster* backend, so `download()` writes a GeoTIFF to disk and returns its path.

## Setup

Imports up front: `tempfile` for a scratch download directory, `numpy` /
`matplotlib` for the plot, `pyramids`' `Dataset` for reading the GeoTIFF, and
`earthlens` for the unified `EarthLens` entry point.

In [ ]:
import tempfile

import matplotlib.pyplot as plt
import numpy as np
from earthlens.core import EarthLens
from pyramids.dataset import Dataset

## Download the building-height tile

We build the GHSL request first — source, the `built_height` variable, the 2018
year window, and a bounding box over Lisbon. The output goes to a temporary
directory created with `tempfile.mkdtemp()`.

In [ ]:
out = tempfile.mkdtemp()
ghsl = EarthLens(
    data_source='ghsl',
    variables=['built_height'],
    start='2018-01-01',
    end='2018-12-31',
    aoi=[-9.20, 38.68, -9.10, 38.78],
    path=out,
)

Downloading is kept as its own step: `download()` fetches the tile, crops it to
the AOI, and returns the list of written GeoTIFF paths.

In [ ]:
paths = ghsl.download(progress_bar=False)
paths

## Read and clean the height array

Read the GeoTIFF into a NumPy array with pyramids' `Dataset`, squeeze the single
band, and mask the no-data sentinel (negative values) to `NaN` so they render as
blank rather than skewing the colour scale.

In [ ]:
raster = Dataset.read_file(str(paths[0]))
arr = np.asarray(raster.read_array(), dtype='float32')
arr = arr[0] if arr.ndim == 3 else arr
arr = np.where(arr < 0, np.nan, arr)

## Plot the building heights

Render the average net building height over Lisbon with a `viridis` colour map.

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(arr, cmap='viridis')
plt.colorbar(label='mean height (m)')
plt.title('GHS-BUILT-H ANBH 2018 (100 m) — Lisbon')
plt.axis('off')
plt.show()